# Text Training: Model Pool and Prediction Generation

Trains the fixed 15-model text pool (6 transformer encoders, 5 lexical baselines,
4 neural non-transformers) on one dataset (here MultiNLI) and writes the
prediction JSON consumed by the experiment pipeline.

Transformer fine-tuning: AdamW lr 2e-5, weight decay 0.01, linear warmup 0.10,
max 4 epochs, max sequence length 256, batch size 16, FP16, early stopping
(patience 2) on validation macro-F1; class-weighted cross-entropy. Sizing and
splits follow the paper. Output schema matches src/io_utils.py.


In [ ]:
# ============================================================
# CELL 1 — TEXT MODEL POOL (15) + LOAD HF MODELS NOW
# (No BaseDir, no smoke test, no extra metadata)
# ============================================================


import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# 15 total entries:
#   - 6 HF transformers
#   - 5 classical / lexical baselines
#   - 4 neural non-transformers
# ------------------------------------------------------------
TEXT_MODEL_POOL = [
    # --- HF transformers (6) ---
    "microsoft/deberta-v3-large",
    "roberta-large",
    "bert-base-cased",
    "distilbert-base-uncased",
    "google/electra-base-discriminator",
    "xlnet-base-cased",

    # --- classical / lexical baselines (5) ---
    "tfidf_logreg_l2",
    "tfidf_linear_svm",
    "char_tfidf_linear_svm",
    "nbsvm",
    "hashing_sgd_hinge",

    # --- neural non-transformers (4) ---
    "fasttext",
    "cnn_text",
    "bilstm",
    "bow_glove_logreg",
]

assert len(TEXT_MODEL_POOL) == 15

# Separate HF vs baselines (used in Cells 4–5)
HF_MODELS = [
    m for m in TEXT_MODEL_POOL
    if not (
        m.startswith("tfidf_") or
        m.startswith("char_tfidf_") or
        m in {"nbsvm", "hashing_sgd_hinge", "fasttext", "cnn_text", "bilstm", "bow_glove_logreg"}
    )
]
BASELINE_MODELS = [m for m in TEXT_MODEL_POOL if m not in HF_MODELS]

print("Text model pool size:", len(TEXT_MODEL_POOL))
print("HF models:", len(HF_MODELS))
print("Baselines:", len(BASELINE_MODELS))

# ------------------------------------------------------------
# Load HF tokenizers + models NOW (fail early)
# NOTE: num_labels is a placeholder; Cell 4 will re-init heads
# ------------------------------------------------------------
tokenizers = {}
preloaded_hf_models = {}
load_errors = {}

print("\n=== LOADING HF MODELS NOW ===")
for name in HF_MODELS:
    print("\n" + "-" * 90)
    print("[LOAD]", name)
    try:
        tok = AutoTokenizer.from_pretrained(name, use_fast=True)
        model = AutoModelForSequenceClassification.from_pretrained(
            name,
            num_labels=4,                  # placeholder (overridden in Cell 4)
            ignore_mismatched_sizes=True
        ).to(DEVICE)

        tokenizers[name] = tok
        preloaded_hf_models[name] = model
        print("  ✅ loaded tokenizer + model")
    except Exception as e:
        load_errors[name] = repr(e)
        print("  ❌ FAILED:", repr(e))

print("\n=== LOAD SUMMARY ===")
print(f"Loaded HF models: {len(preloaded_hf_models)}/{len(HF_MODELS)}")
if len(load_errors) > 0:
    print("\nFailures:")
    for k, v in load_errors.items():
        print(" -", k, "->", v)

print("\nCELL 1 DONE.")


In [ ]:
# ============================================================
# CELL 2 — TEXT DATASET — RATIONALITY CHECK ONLY
# Dataset: MultiNLI (3-way NLI: entailment/neutral/contradiction)
# We use official TRAIN + VALIDATION_MATCHED (has labels).
# Labels are REMAPPED to contiguous 0..C-1
# NOTE: Text is a *pair* -> we store as a single combined string with a clear separator.
# ============================================================

import numpy as np
from collections import Counter
from datasets import load_dataset

RANDOM_STATE = 42

# ----------------------------
# Load dataset (robust)
# ----------------------------
ds = None
last_err = None
for ds_name in ["multi_nli"]:
    try:
        ds = load_dataset(ds_name)
        print("Loaded dataset: multi_nli")
        break
    except Exception as e:
        last_err = e

if ds is None:
    raise RuntimeError(f"Failed to load MultiNLI dataset. Last error: {repr(last_err)}")

# ----------------------------
# Pick labeled splits:
#  - train
#  - validation_matched  (labeled)
# (test is unlabeled in MultiNLI)
# ----------------------------
if "train" not in ds:
    raise RuntimeError(f"Missing 'train' split. Available splits: {list(ds.keys())}")

val_key = None
for cand in ["validation_matched", "validation_matched"]:
    if cand in ds:
        val_key = cand
        break
if val_key is None:
    raise RuntimeError(f"Missing validation_matched split. Available splits: {list(ds.keys())}")

sp_train = ds["train"]
sp_val   = ds[val_key]

# ----------------------------
# Detect columns
# Common MultiNLI:
#   - premise
#   - hypothesis
#   - label
# ----------------------------
train_cols = sp_train.column_names

prem_col = "premise" if "premise" in train_cols else None
hyp_col  = "hypothesis" if "hypothesis" in train_cols else None
label_col = "label" if "label" in train_cols else None

if prem_col is None or hyp_col is None or label_col is None:
    raise RuntimeError(f"Could not detect premise/hypothesis/label columns. train columns={train_cols}")

# ----------------------------
# Pool train + validation
# ----------------------------
premises = list(sp_train[prem_col]) + list(sp_val[prem_col])
hypos    = list(sp_train[hyp_col])  + list(sp_val[hyp_col])
y_raw    = list(sp_train[label_col]) + list(sp_val[label_col])

premises = np.asarray(premises, dtype=object)
hypos    = np.asarray(hypos,    dtype=object)
y_raw    = np.asarray(y_raw)

# ----------------------------
# Remove invalid labels if any (MultiNLI sometimes has -1 in some variants)
# ----------------------------
valid_mask = np.asarray(y_raw, dtype=np.int64) >= 0
premises = premises[valid_mask]
hypos    = hypos[valid_mask]
y_raw    = y_raw[valid_mask]

# ----------------------------
# Build single-text inputs (keeps Cell 4 unchanged)
# ----------------------------
SEP = " [SEP] "
texts = np.asarray([str(p) + SEP + str(h) for p, h in zip(premises, hypos)], dtype=object)

# ----------------------------
# 🔑 LABEL NORMALIZATION
# ----------------------------
unique_labels = sorted(np.unique(y_raw).tolist())
label_map = {lab: i for i, lab in enumerate(unique_labels)}
y = np.asarray([label_map[v] for v in y_raw], dtype=np.int64)

num_classes = int(len(unique_labels))

# ----------------------------
# Stats
# ----------------------------
N_total = int(len(y))
counts = Counter(y.tolist())
freqs = np.array([counts[i] for i in range(num_classes)], dtype=np.int64)
p_min = float(freqs.min() / N_total)

# ----------------------------
# Size constraints (UNCHANGED)
# ----------------------------
N_test_min = int(np.ceil(20 / p_min))
N_test_target = 10 * N_test_min

N_train_min = int(np.ceil(50 / p_min))
N_train_max = int(np.ceil(500 / p_min))

N_train = int(min(N_train_max, N_total - N_test_target))

print("=== DATASET SUMMARY ===")
print("Dataset: MultiNLI (pooled train + validation_matched)")
print(f"Premise col: {prem_col} | Hypothesis col: {hyp_col} | Label col: {label_col}")
print(f"Input format: premise + '{SEP.strip()}' + hypothesis")
print(f"N_total: {N_total}")
print(f"Num classes: {num_classes}")
print(f"Label counts: {dict((i, int(counts[i])) for i in range(num_classes))}")
print(f"p_min: {p_min:.6f}")
print()

print("=== SIZE CONSTRAINTS ===")
print(f"N_test_min     = {N_test_min}")
print(f"N_test_target  = {N_test_target}")
print(f"N_train_min    = {N_train_min}")
print(f"N_train_max    = {N_train_max}")
print(f"Chosen N_train = {N_train}")
print()

if N_train > N_train_min and (N_train + N_test_target) <= N_total:
    print("✅ DATASET PASSES nTrain / nTest RATIONALITY CHECK")
else:
    print("❌ DATASET FAILS nTrain / nTest RATIONALITY CHECK (exclude from main benchmark)")


In [ ]:
# ============================================================
# CELL 3 — SPLIT (exact sizes) + TEXT_SPLITS
# Fully aligned with AG News / DBpedia / Yahoo / 20NG
# ============================================================

from sklearn.model_selection import train_test_split
import numpy as np

VAL_FRAC = 0.10
RANDOM_STATE = 42

N_TEST_TARGET = int(N_test_target)
N_TRAIN_TOTAL = int(N_train)

# ---- exact test split ----
X_pool, X_test, y_pool, y_test = train_test_split(
    texts, y,
    test_size=N_TEST_TARGET,
    stratify=y,
    random_state=RANDOM_STATE
)

# ---- exact train+val size ----
if N_TRAIN_TOTAL == len(y_pool):
    X_trainval, y_trainval = X_pool, y_pool
else:
    X_trainval, _, y_trainval, _ = train_test_split(
        X_pool, y_pool,
        train_size=N_TRAIN_TOTAL,
        stratify=y_pool,
        random_state=RANDOM_STATE
    )

# ---- train / val split ----
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=VAL_FRAC,
    stratify=y_trainval,
    random_state=RANDOM_STATE
)

def bincount_dict(arr, k):
    bc = np.bincount(np.asarray(arr, dtype=np.int64), minlength=int(k))
    return {int(i): int(bc[i]) for i in range(int(k))}

print("=== SPLITS ===")
print("Dataset: MultiNLI")
print(f"Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}")
print("Class counts:")
print("  Train:", bincount_dict(y_train, num_classes))
print("  Val:  ", bincount_dict(y_val,   num_classes))
print("  Test: ", bincount_dict(y_test,  num_classes))

# label names (metadata only)
label_names = None
try:
    # MultiNLI label names in some versions:
    # 0: entailment, 1: neutral, 2: contradiction
    label_names = ds["train"].features[label_col].names
except Exception:
    # fallback (still correct for typical MultiNLI)
    label_names = ["entailment", "neutral", "contradiction"]

TEXT_SPLITS = {
    "train": (X_train.tolist(), np.asarray(y_train, dtype=np.int64).astype(int).tolist()),
    "val":   (X_val.tolist(),   np.asarray(y_val,   dtype=np.int64).astype(int).tolist()),
    "test":  (X_test.tolist(),  np.asarray(y_test,  dtype=np.int64).astype(int).tolist()),
    "label_names": label_names,
    "num_classes": int(num_classes),
}


In [ ]:
# ============================================================
# CELL 4 — TRAINING (TEXT POOL) — THREE REGIMES (A / B / C)
#
# Regime A: HF Transformers (fine-tuned)
# Regime B: Classical / Lexical baselines (sklearn, single-fit)
# Regime C: Neural non-transformers (trained from scratch, PyTorch)
# ============================================================

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import f1_score

# --- classical baselines ---
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    HashingVectorizer,
    CountVectorizer,
)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# Shared training hyperparameters (A + C)
# ------------------------------------------------------------
MAX_LEN = 256
BATCH_SIZE = 16
MAX_EPOCHS = 4
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.10
GRAD_CLIP = 1.0
EARLY_STOP_PATIENCE = 2

# LR differs by regime
LR_TRANSFORMER = 2e-5   # Regime A
LR_NEURAL      = 1e-3   # Regime C

USE_FP16 = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)

# ------------------------------------------------------------
# Pull splits from Cell 3
# ------------------------------------------------------------
X_train, y_train = TEXT_SPLITS["train"]
X_val,   y_val   = TEXT_SPLITS["val"]
num_classes = int(TEXT_SPLITS["num_classes"])

# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------
def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

# ------------------------------------------------------------
# Class weights (used in A + C)
# ------------------------------------------------------------
y_train_np = np.asarray(y_train, dtype=np.int64)
class_counts = np.bincount(y_train_np, minlength=num_classes).astype(np.float64)
class_weights = class_counts.sum() / (num_classes * np.maximum(class_counts, 1.0))
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

# ============================================================
# REGIME IDENTIFICATION
# ============================================================
REGIME_A_HF = {
    "microsoft/deberta-v3-large",
    "roberta-large",
    "bert-base-cased",
    "distilbert-base-uncased",
    "google/electra-base-discriminator",
    "xlnet-base-cased",
}

REGIME_B_BASELINES = {
    "tfidf_logreg_l2",
    "tfidf_linear_svm",
    "tfidf_nb",
    "char_tfidf_linear_svm",
    "hashing_sgd_hinge",
    "nbsvm",
}

REGIME_C_NEURAL = {
    "fasttext",
    "cnn_text",
    "bilstm",
    "bow_glove_logreg",
}

# ============================================================
# REGIME B — CLASSICAL BASELINES (unchanged logic)
# ============================================================
def make_baseline_pipeline(name: str):
    word_tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95)
    char_tfidf = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=2, max_df=0.95)

    if name == "tfidf_logreg_l2":
        return Pipeline([("vec", word_tfidf), ("clf", LogisticRegression(max_iter=2000))])
    if name == "tfidf_linear_svm":
        return Pipeline([("vec", word_tfidf), ("clf", LinearSVC())])
    if name == "tfidf_nb":
        return Pipeline([("vec", word_tfidf), ("clf", MultinomialNB(alpha=0.5))])
    if name == "char_tfidf_linear_svm":
        return Pipeline([("vec", char_tfidf), ("clf", LinearSVC())])
    if name == "hashing_sgd_hinge":
        return Pipeline([
            ("vec", HashingVectorizer(ngram_range=(1,2), n_features=2**18, alternate_sign=False)),
            ("clf", SGDClassifier(loss="hinge", alpha=1e-5)),
        ])
    if name == "nbsvm":
        # practical NB-SVM fallback (multi-class safe)
        return Pipeline([
            ("vec", CountVectorizer(binary=True, min_df=2)),
            ("clf", LinearSVC()),
        ])

    raise ValueError(name)

# ============================================================
# REGIME C — NEURAL NON-TRANSFORMERS (from scratch)
# ============================================================
class SimpleTokenizer:
    def __init__(self, texts, max_vocab=50000):
        from collections import Counter
        counter = Counter()
        for t in texts:
            counter.update(t.split())
        vocab = ["<pad>", "<unk>"] + [w for w,_ in counter.most_common(max_vocab)]
        self.stoi = {w:i for i,w in enumerate(vocab)}
        self.pad_id = 0
        self.unk_id = 1

    def encode(self, text, max_len):
        ids = [self.stoi.get(w, self.unk_id) for w in text.split()]
        ids = ids[:max_len]
        if len(ids) < max_len:
            ids += [self.pad_id] * (max_len - len(ids))
        return ids

class NeuralTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(
                self.tokenizer.encode(self.texts[idx], MAX_LEN),
                dtype=torch.long,
            ),
            "labels": torch.tensor(int(self.labels[idx]), dtype=torch.long),
        }

class FastTextModel(nn.Module):
    def __init__(self, vocab, num_classes, emb_dim=300):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb_dim, padding_idx=0)
        self.fc = nn.Linear(emb_dim, num_classes)

    def forward(self, input_ids):
        x = self.emb(input_ids).mean(dim=1)
        return self.fc(x)

class TextCNN(nn.Module):
    def __init__(self, vocab, num_classes, emb_dim=300):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(emb_dim, 128, k) for k in (3,4,5)
        ])
        self.fc = nn.Linear(128*3, num_classes)

    def forward(self, input_ids):
        x = self.emb(input_ids).transpose(1,2)
        feats = [torch.max(torch.relu(c(x)), dim=2)[0] for c in self.convs]
        return self.fc(torch.cat(feats, dim=1))

class BiLSTM(nn.Module):
    def __init__(self, vocab, num_classes, emb_dim=300, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, num_classes)

    def forward(self, input_ids):
        x = self.emb(input_ids)
        out,_ = self.lstm(x)
        pooled = out.mean(dim=1)
        return self.fc(pooled)

# ============================================================
# TRAINING CONTAINERS
# ============================================================
trained_text_models = {}
train_logs = {}

# ============================================================
# MAIN TRAIN LOOP (ALL REGIMES)
# ============================================================
for model_name in TEXT_MODEL_POOL:
    print("\n" + "="*90)
    print("[TRAIN TEXT]", model_name)

    # ================= REGIME B =================
    if model_name in REGIME_B_BASELINES:
        pipe = make_baseline_pipeline(model_name)
        pipe.fit(X_train, y_train)
        val_f1 = float(f1_score(y_val, pipe.predict(X_val), average="macro"))
        trained_text_models[model_name] = pipe
        train_logs[model_name] = {"regime": "B", "val_macro_f1": val_f1}
        print(f"  -> Regime B | val_macro_f1={val_f1:.4f}")
        continue

    # ================= REGIME A =================
    if model_name in REGIME_A_HF:
        cleanup_cuda()
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=num_classes, ignore_mismatched_sizes=True
        ).to(DEVICE)

        def make_loader(X, y, shuffle):
            class HFDS(Dataset):
                def __len__(self): return len(X)
                def __getitem__(self, i):
                    enc = tok(X[i], truncation=True, padding="max_length",
                              max_length=MAX_LEN, return_tensors="pt")
                    return {
                        "input_ids": enc["input_ids"].squeeze(0),
                        "attention_mask": enc["attention_mask"].squeeze(0),
                        "labels": torch.tensor(int(y[i]), dtype=torch.long),
                    }
            return DataLoader(HFDS(), batch_size=BATCH_SIZE, shuffle=shuffle)

        train_loader = make_loader(X_train, y_train, True)
        val_loader   = make_loader(X_val,   y_val,   False)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR_TRANSFORMER, weight_decay=WEIGHT_DECAY)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            int(WARMUP_FRAC * len(train_loader) * MAX_EPOCHS),
            len(train_loader) * MAX_EPOCHS,
        )
        criterion = nn.CrossEntropyLoss(weight=class_weights_t)

        best_f1, best_state, patience = -1, None, 0
        for _ in range(MAX_EPOCHS):
            model.train()
            for b in train_loader:
                b = {k:v.to(DEVICE) for k,v in b.items()}
                optimizer.zero_grad()
                with torch.cuda.amp.autocast(enabled=USE_FP16):
                    loss = criterion(model(**b).logits, b["labels"])
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update(); scheduler.step()

            model.eval()
            preds, trues = [], []
            for b in val_loader:
                b = {k:v.to(DEVICE) for k,v in b.items()}
                preds.append(model(**b).logits.argmax(1).cpu().numpy())
                trues.append(b["labels"].cpu().numpy())
            f1 = float(f1_score(np.concatenate(trues), np.concatenate(preds), average="macro"))

            if f1 > best_f1:
                best_f1, best_state, patience = f1, model.state_dict(), 0
            else:
                patience += 1
                if patience >= EARLY_STOP_PATIENCE:
                    break

        model.load_state_dict(best_state)
        trained_text_models[model_name] = model
        train_logs[model_name] = {"regime": "A", "val_macro_f1": best_f1}
        print(f"  -> Regime A | best_val_macro_f1={best_f1:.4f}")
        continue

    # ================= REGIME C =================
    if model_name in REGIME_C_NEURAL:
        cleanup_cuda()
        tok = SimpleTokenizer(X_train)
        train_ds = NeuralTextDataset(X_train, y_train, tok)
        val_ds   = NeuralTextDataset(X_val,   y_val,   tok)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

        if model_name == "fasttext":
            model = FastTextModel(len(tok.stoi), num_classes)
        elif model_name == "cnn_text":
            model = TextCNN(len(tok.stoi), num_classes)
        else:
            model = BiLSTM(len(tok.stoi), num_classes)

        model = model.to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR_NEURAL, weight_decay=WEIGHT_DECAY)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            int(WARMUP_FRAC * len(train_loader) * MAX_EPOCHS),
            len(train_loader) * MAX_EPOCHS,
        )
        criterion = nn.CrossEntropyLoss(weight=class_weights_t)

        best_f1, best_state, patience = -1, None, 0
        for _ in range(MAX_EPOCHS):
            model.train()
            for b in train_loader:
                optimizer.zero_grad()
                logits = model(b["input_ids"].to(DEVICE))
                loss = criterion(logits, b["labels"].to(DEVICE))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()

            model.eval()
            preds, trues = [], []
            for b in val_loader:
                logits = model(b["input_ids"].to(DEVICE))
                preds.append(logits.argmax(1).cpu().numpy())
                trues.append(b["labels"].numpy())
            f1 = float(f1_score(np.concatenate(trues), np.concatenate(preds), average="macro"))

            if f1 > best_f1:
                best_f1, best_state, patience = f1, model.state_dict(), 0
            else:
                patience += 1
                if patience >= EARLY_STOP_PATIENCE:
                    break

        model.load_state_dict(best_state)
        trained_text_models[model_name] = model
        train_logs[model_name] = {"regime": "C", "val_macro_f1": best_f1}
        print(f"  -> Regime C | best_val_macro_f1={best_f1:.4f}")

print("\nCELL 4 DONE — All regimes trained.")


In [ ]:
# ============================================================
# TEXT CELL 5 — EVAL + PRINT PERFORMANCE + SAVE JSON TO DRIVE
# UPDATED: supports Regime A (HF) + Regime B (lexical baselines) + Regime C (neural scratch)
# - For HF: store pre-softmax logits
# - For baselines: store uncalibrated score vectors under "logits"
# - For Regime C: store raw classifier logits under "logits"
# ============================================================

import os, json
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

DATASET_NAME = "MultiNLI"

BASE_DIR = "data/predictions"
OUT_DIR = os.path.join(BASE_DIR, DATASET_NAME)
os.makedirs(OUT_DIR, exist_ok=True)

OUT_JSON = os.path.join(OUT_DIR, f"{DATASET_NAME}_test_predictions_logits.json")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

X_test, y_test = TEXT_SPLITS["test"]
y_test_np = np.asarray(y_test, dtype=np.int64)
NUM_CLASSES = int(TEXT_SPLITS["num_classes"])

# ----------------------------
# HF Test Dataset (Regime A only)
# ----------------------------
class TestDatasetHF(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        return item, int(self.labels[idx])

# ----------------------------
# Regime C Test Dataset (neural scratch)
# Uses the SimpleTokenizer from Cell 4 (trained on X_train)
# ----------------------------
class TestDatasetNeural(torch.utils.data.Dataset):
    def __init__(self, texts, labels, simple_tokenizer):
        self.texts = texts
        self.labels = labels
        self.tok = simple_tokenizer
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.tok.encode(self.texts[idx], 256), dtype=torch.long),
            "labels": torch.tensor(int(self.labels[idx]), dtype=torch.long),
        }

# ----------------------------
# Alignment assertion
# ----------------------------
y_check = np.array([int(y_test[i]) for i in range(len(y_test))], dtype=np.int64)
assert np.array_equal(y_check, y_test_np), "Mismatch: y_test order != test order"

# ----------------------------
# Helpers: model family detection
# ----------------------------
def is_baseline_name(name: str) -> bool:
    return name in {
        "tfidf_logreg_l2",
        "tfidf_linear_svm",
        "tfidf_nb",
        "char_tfidf_linear_svm",
        "hashing_sgd_hinge",
        "nbsvm",
    }

def is_neural_name(name: str) -> bool:
    return name in {
        "fasttext",
        "cnn_text",
        "bilstm",
        "bow_glove_logreg",
    }

# ----------------------------
# HF logits (Regime A)
# ----------------------------
@torch.no_grad()
def predict_logits_hf(model, tokenizer):
    model.eval()
    loader = DataLoader(
        TestDatasetHF(X_test, y_test, tokenizer, 256),
        batch_size=32,
        shuffle=False
    )
    chunks = []
    for batch, _ in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch).logits
        chunks.append(logits.detach().cpu().numpy())
    return np.concatenate(chunks, axis=0)

# ----------------------------
# Baseline scores (Regime B)
# ----------------------------
def scores_baseline(pipe, num_classes: int):
    """
    Return (scores, pred):
      - scores: shape [N, C] float, uncalibrated score vectors
      - pred:   argmax over scores
    Stored under "logits" for compatibility with MVLogit.
    """
    if hasattr(pipe, "decision_function"):
        s = np.asarray(pipe.decision_function(X_test))
        if s.ndim == 1:
            s = np.stack([-s, s], axis=1)
        if s.shape[1] != num_classes:
            raise ValueError(f"decision_function returned shape {s.shape}, expected second dim={num_classes}")
        pred = s.argmax(axis=1)
        return s.astype(np.float32), pred.astype(np.int64)

    if hasattr(pipe, "predict_proba"):
        p = np.asarray(pipe.predict_proba(X_test))
        if p.shape[1] != num_classes:
            raise ValueError(f"predict_proba returned shape {p.shape}, expected second dim={num_classes}")
        eps = 1e-12
        s = np.log(np.clip(p, eps, 1.0))
        pred = s.argmax(axis=1)
        return s.astype(np.float32), pred.astype(np.int64)

    pred = np.asarray(pipe.predict(X_test), dtype=np.int64)
    s = np.full((len(pred), num_classes), -1.0, dtype=np.float32)
    s[np.arange(len(pred)), pred] = 0.0
    return s, pred

# ----------------------------
# Neural logits (Regime C)
# IMPORTANT: In Cell 4, the Regime C tokenizer is created inside the loop.
# We therefore recreate it here from X_train to keep it consistent & leakage-free.
# (Same exact construction: SimpleTokenizer(X_train))
# ----------------------------
@torch.no_grad()
def predict_logits_neural(model, simple_tokenizer):
    model.eval()
    loader = DataLoader(
        TestDatasetNeural(X_test, y_test, simple_tokenizer),
        batch_size=32,
        shuffle=False
    )
    chunks = []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        logits = model(input_ids)
        chunks.append(logits.detach().cpu().numpy())
    return np.concatenate(chunks, axis=0)

# ----------------------------
# Evaluate + collect
# ----------------------------
results = {
    "dataset": DATASET_NAME,
    "num_classes": int(NUM_CLASSES),
    "N_test_min": int(N_test_min),
    "N_test_target": int(N_test_target),
    "y_test": y_test_np.tolist(),
    "models": {}
}

print("=== TEST PERFORMANCE (accuracy, macro_f1) ===")

# stable print order
model_names = list(TEXT_MODEL_POOL) if "TEXT_MODEL_POOL" in globals() else list(trained_text_models.keys())

# Rebuild the Regime C tokenizer (same as Cell 4)
# NOTE: This assumes X_train is available in globals (it is in Cell 4).
simple_tok = None
if "X_train" in globals():
    try:
        simple_tok = SimpleTokenizer(X_train)
    except Exception:
        simple_tok = None

for model_name in model_names:
    model = trained_text_models[model_name]

    # ---------------- Regime B: baselines ----------------
    if is_baseline_name(model_name):
        logits, pred = scores_baseline(model, NUM_CLASSES)
        acc = float(accuracy_score(y_test_np, pred))
        mf1 = float(f1_score(y_test_np, pred, average="macro"))

        print(f"{model_name:40s}  acc={acc:.4f}  macro_f1={mf1:.4f}")

        results["models"][model_name] = {
            "logits": logits.tolist(),
            "y_pred": pred.tolist(),
            "metrics": {"accuracy": acc, "macro_f1": mf1},
            "kind": "baseline",
        }
        continue

    # ---------------- Regime C: neural scratch ----------------
    if is_neural_name(model_name):
        if simple_tok is None:
            raise RuntimeError(
                "Regime C requires SimpleTokenizer(X_train) but X_train was not found in globals. "
                "Make sure Cell 4 ran in the same runtime before Cell 5."
            )

        logits = predict_logits_neural(model, simple_tok)
        pred = logits.argmax(axis=1).astype(np.int64)

        acc = float(accuracy_score(y_test_np, pred))
        mf1 = float(f1_score(y_test_np, pred, average="macro"))

        print(f"{model_name:40s}  acc={acc:.4f}  macro_f1={mf1:.4f}")

        results["models"][model_name] = {
            "logits": logits.tolist(),
            "y_pred": pred.tolist(),
            "metrics": {"accuracy": acc, "macro_f1": mf1},
            "kind": "neural_scratch",
        }
        continue

    # ---------------- Regime A: HF models ----------------
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True) if model_name not in tokenizers else tokenizers[model_name]
    logits = predict_logits_hf(model, tok)
    pred = logits.argmax(axis=1).astype(np.int64)

    acc = float(accuracy_score(y_test_np, pred))
    mf1 = float(f1_score(y_test_np, pred, average="macro"))

    print(f"{model_name:40s}  acc={acc:.4f}  macro_f1={mf1:.4f}")

    results["models"][model_name] = {
        "logits": logits.tolist(),
        "y_pred": pred.tolist(),
        "metrics": {"accuracy": acc, "macro_f1": mf1},
        "kind": "hf",
    }

print("\nModels evaluated:", len(results["models"]))
print("Saving to:", OUT_JSON)

with open(OUT_JSON, "w") as f:
    json.dump(results, f)

print("✅ Saved.")
print("✅ True labels are included in JSON under key: 'y_test'")
print("✅ Test size range included in JSON under keys: 'N_test_min', 'N_test_target'")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Assumes you have ONE of these in memory:
#  1) results["models"][name]["y_pred"]  (from our JSON export step)
#  2) all_predictions_pred[name] = np.array([...])
#  3) all_predictions_logits[name] = np.array([N_test, C])  -> will argmax to y_pred
# ============================================================

# ---- Resolve predictions dict ----
if "all_predictions_pred" in globals():
    pred_dict = {k: np.asarray(v) for k, v in all_predictions_pred.items()}
elif "results" in globals() and isinstance(results, dict) and "models" in results:
    pred_dict = {k: np.asarray(v["y_pred"]) for k, v in results["models"].items()}
elif "all_predictions_logits" in globals():
    pred_dict = {k: np.asarray(v).argmax(axis=1) for k, v in all_predictions_logits.items()}
else:
    raise RuntimeError(
        "No predictions found. Expected all_predictions_pred, or results['models'][..]['y_pred'], "
        "or all_predictions_logits."
    )

model_names = sorted(pred_dict.keys())
preds = np.stack([pred_dict[m] for m in model_names], axis=0)  # (M, N)

M, N = preds.shape
disagree = np.zeros((M, M), dtype=np.float64)

# ============================================================
# 1) Pairwise disagreement matrix (fraction different labels)
# ============================================================
for i in range(M):
    for j in range(M):
        disagree[i, j] = np.mean(preds[i] != preds[j])

# ============================================================
# 2) Off-diagonal summary stats
# ============================================================
off_mask = ~np.eye(M, dtype=bool)
off_vals = disagree[off_mask]

print("=== Off-diagonal disagreement stats ===")
print(f"num_models: {M}, num_samples: {N}")
print(f"mean: {off_vals.mean():.6f}")
print(f"std:  {off_vals.std(ddof=0):.6f}")
print(f"min:  {off_vals.min():.6f}")
print(f"max:  {off_vals.max():.6f}")

# ============================================================
# 3) Plots
#  - heatmap of disagreement matrix
#  - histogram of off-diagonal disagreement values
# ============================================================

plt.figure(figsize=(max(8, 0.45 * M), max(6, 0.45 * M)))
plt.imshow(disagree, aspect="auto")
plt.colorbar(label="Disagreement (fraction different labels)")
plt.xticks(range(M), model_names, rotation=90)
plt.yticks(range(M), model_names)
plt.title("Pairwise Disagreement Matrix")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.5))
plt.hist(off_vals, bins=30)
plt.title("Histogram of Off-diagonal Pairwise Disagreements")
plt.xlabel("Disagreement (fraction different labels)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


icially force gains when meaningful expert diversity is